MINGGU 1

Reading and Exploring Data with Python & SQLite

Dataset: Titanic Dataset (Kaggle)

DATASET

Dataset yang digunakan adalah Titanic Dataset dari Kaggle. Dataset berisi informasi penumpang kapal Titanic, seperti kelas penumpang, jenis kelamin, usia, tarif, jumlah keluarga, dan status keselamatan.

Download dataset dari Kaggle, lalu simpan file CSV di folder `data` dengan nama `titanic.csv`.


STRUKTUR FOLDER

```text
data-mining-weekly/
├── data/
│   └── titanic.csv
└── titanic_data_mining.ipynb
```


In [1]:
import sqlite3 as sq3
import pandas as pd
import os

1. Membaca Dataset dengan Pandas

In [2]:
csv_path = 'data/titanic.csv'

titanic = pd.read_csv(csv_path)

titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
print('Jumlah baris dan kolom:', titanic.shape)

Jumlah baris dan kolom: (891, 12)


In [17]:
titanic.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


2. Pemeriksaan Data Awal

In [18]:
titanic.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [19]:
titanic.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


3. Membuat Database SQLite

In [22]:
#Membuat folder data jika belum tersedia
os.makedirs('data', exist_ok=True)

#Membuat koneksi SQLite
db_path = 'data/titanic.db'
con = sq3.connect(db_path)

#Menyimpan DataFrame ke tabel SQLite
titanic.to_sql('passengers', con, if_exists='replace', index=False)

print('Database berhasil dibuat:', db_path)

Database berhasil dibuat: data/titanic.db


In [23]:
#Memastikan tabel tersedia
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    con
)
tables

,name
0,passengers


4. Membaca Data dari SQLite

In [24]:
query = '''
SELECT *
FROM passengers;
'''

observations = pd.read_sql(query, con)
observations.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


5. Query SQL untuk Eksplorasi Data

In [25]:
#Jumlah penumpang berdasarkan kelas
query = '''
SELECT Pclass,
       COUNT(*) AS jumlah_penumpang
FROM passengers
GROUP BY Pclass
ORDER BY Pclass;
'''

class_summary = pd.read_sql(query, con)
class_summary

,Pclass,jumlah_penumpang
0,1,216
1,2,184
2,3,491


In [26]:
#Rata-rata usia berdasarkan jenis kelamin
query = '''
SELECT Sex,
       COUNT(*) AS jumlah_penumpang,
       AVG(Age) AS rata_rata_usia
FROM passengers
GROUP BY Sex;
'''

gender_summary = pd.read_sql(query, con)
gender_summary

,Sex,jumlah_penumpang,rata_rata_usia
0,female,314,27.915709
1,male,577,30.726645


In [27]:
#Tingkat keselamatan berdasarkan kelas penumpang
query = '''
SELECT Pclass,
       COUNT(*) AS total_penumpang,
       SUM(Survived) AS total_selamat,
       AVG(Survived) AS survival_rate
FROM passengers
GROUP BY Pclass
ORDER BY survival_rate DESC;
'''

survival_summary = pd.read_sql(query, con)
survival_summary

,Pclass,total_penumpang,total_selamat,survival_rate
0,1,216,136,0.629630
1,2,184,87,0.472826
2,3,491,119,0.242363


6. Membandingkan SQL dan Pandas

Analisis yang sama dapat dilakukan menggunakan Pandas `groupby()`.


In [28]:
#Analisis yang sama menggunakan Pandas
pandas_summary = (
    titanic.groupby('Pclass')
    .agg(
        total_penumpang=('PassengerId', 'count'),
        total_selamat=('Survived', 'sum'),
        survival_rate=('Survived', 'mean')
    )
    .reset_index()
)

pandas_summary

,Pclass,total_penumpang,total_selamat,survival_rate
0,1,216,136,0.629630
1,2,184,87,0.472826
2,3,491,119,0.242363


In [29]:
#Menutup koneksi database setelah selesai
con.close()